# Cell 1 - verify GPU is enabled
This checks that the Colab runtime has a GPU available.
Embedding models run much faster on GPU

In [ ]:
import torch
assert torch.cuda_is_available(), "Switch to GPU runtime first"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Cell 2 - Clone or update Github Repo and install dependencies
This cell ensures the latest backend code is loaded from Github and installs all required libraries.

In [ ]:
import os
import subprocess

REPO = "https://github.com/tanveer-12/personal-knowledge-base.git"
DIR = "/content/personal-knowledge-base"

if not os.path.exists(DIR):
    print("Cloning Repository...")
    subprocess.run(["git", "clone", REPO, DIR], check=True)
else:
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", DIR, "pull"], check=True)

os.chdir(f"{DIR}/backend")
print("Current Directory: ", os.getcwd())

# Step 1 — Upgrade pip
print("Upgrading pip...")
subprocess.run(["pip", "install", "--upgrade", "pip"], check=True)

# Step 2 — Install core backend packages
print("Installing FastAPI, Uvicorn, SQLAlchemy, database libraries, pyngrok...")
subprocess.run([
    "pip", "install", "-q",
    "fastapi", "uvicorn", "sqlalchemy", "psycopg2-binary", "python-dotenv", "httpx","pyngrok"
], check=True)

# Step 3 — Install embeddings packages (sentence-transformers + torch)
# Colab already has a GPU version of torch; we install sentence-transformers only
print("Installing SentenceTransformers...")
subprocess.run([
    "pip", "install", "-q", "sentence-transformers", "torch"
], check=True)

# Step 4 — Install pgvector
print("Installing pgvector...")
subprocess.run([
    "pip", "install", "-q", "pgvector"
], check=True)

print("Dependencies installed successfully!")

# Cell 3 - Load Environment variables and initialize database
This cell loads the database connection string from Colab Secrets, verifies the connection, and runs the database initialization script if needed

In [ ]:
import os
import psycopg2
from google.colab import userdata

# Load secrets from Colab
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
os.environ["APP_ENV"] = "development"

print("Environment variables loaded.")

# Verify database connection
print("Connecting to database...")

conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()

cur.execute("SELECT version();")
version = cur.fetchone()[0]

print("PostgreSQL version: ", version[:60], "...")

cur.close()
conn.close()

print("Database connection successful.")

# Run init.sql to ensure tables and indexes exist
INIT_SQL_PATH = "../database/init.sql"

print("Running database intialization script...")

with open(INIT_SQL_PATH, "r") as f:
    sql = f.read()

conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()

cur.execute(sql)
conn.commit()

cur.close()
conn.close()

print("Database initialization coomplete.")

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok

# Kill any existing ngrok tunnels
ngrok.kill()

# Start FastAPI server in background
print("Starting FastAPI server on port 8000...")
uvicorn_process = subprocess.Popen([
    "uvicorn",
    "app.main:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])

# Wait a few seconds to allow model to load (first load ~80MB)
print("Waiting for model to load (10s)...")
time.sleep(10)

# Open ngrok tunnel
public_url = ngrok.connect(8000)
print("=== SERVER READY ===")
print(f"API URL: {public_url}")
print(f"Docs:    {public_url}/docs")
print(f"Paste into Next.js .env.local:\nNEXT_PUBLIC_API_URL={public_url}")

In [ ]:
# CELL 5 — Verify FastAPI server health
import requests
import os

# Use the ngrok public URL from Cell 4
URL = os.environ.get("PUBLIC_API_URL", None)
if not URL:
    URL = input("Enter your ngrok public URL (from Cell 4): ")

try:
    r = requests.get(f"{URL}/health")
    if r.status_code == 200:
        print("=== SERVER HEALTH ===")
        data = r.json()
        print(f"Environment: {data.get('env')}")
        print(f"Embedding model: {data.get('model')}")
        print(f"Embedding dimensions: {data.get('dims')}")
    else:
        print(f"Server returned status code: {r.status_code}")
except Exception as e:
    print("Failed to connect to server:", e)